In [1]:
"""
Federated Learning — FedProx with Neural Network (PyTorch)
Dataset: ULB Credit Card Fraud (creditcard.csv from Kaggle)
Adds proximal term mu * ||w - w_global||^2 to combat non-IID drift
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, recall_score
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import copy, warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_BANKS       = 5
FL_ROUNDS     = 20
LOCAL_EPOCHS  = 5
BATCH_SIZE    = 256
LR            = 0.001
MU            = 0.01     # FedProx proximal strength (key hyperparameter)
RANDOM_STATE  = 42
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
print("Loading dataset...")
df = pd.read_csv("/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv")
X  = df.drop("Class", axis=1).values
y  = df["Class"].values

scaler  = StandardScaler()
X       = scaler.fit_transform(X)
N_FEAT  = X.shape[1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# ── NON-IID SPLIT ─────────────────────────────────────────────────────────────
def non_iid_split(X, y, n, seed=42):
    rng = np.random.default_rng(seed)
    fraud_idx = np.where(y == 1)[0]
    legit_idx = np.where(y == 0)[0]
    splits    = rng.dirichlet(np.ones(n) * 0.4) * len(fraud_idx)
    splits    = np.round(splits).astype(int)
    splits[-1]= len(fraud_idx) - splits[:-1].sum()
    banks, fi, li = [], 0, len(legit_idx) // n
    for i in range(n):
        fe = fi + splits[i]
        le = (i + 1) * (len(legit_idx) // n)
        idx = np.concatenate([fraud_idx[fi:fe], legit_idx[i*li:le]])
        rng.shuffle(idx)
        banks.append((X[idx], y[idx]))
        fi = fe
    return banks

print(f"\nNon-IID split across {N_BANKS} banks:")
bank_data = non_iid_split(X_train, y_train, N_BANKS)
for i, (xb, yb) in enumerate(bank_data):
    print(f"  Bank {i+1}: {len(xb):>6,} records | fraud {yb.mean()*100:.2f}%")

# ── MODEL ─────────────────────────────────────────────────────────────────────
class FraudNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze()

def make_loader(X, y, batch_size=BATCH_SIZE):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.float32)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

# ── FEDPROX LOCAL TRAIN ───────────────────────────────────────────────────────
def local_train_fedprox(model, global_model, loader, mu=MU):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCELoss()

    # Class weight for imbalance
    for X_b, y_b in loader:
        pos_weight = (y_b == 0).sum() / max((y_b == 1).sum(), 1)
        break

    for _ in range(LOCAL_EPOCHS):
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            preds = model(X_b)

            # Standard loss
            loss = criterion(preds, y_b)

            # Proximal term: mu/2 * ||w - w_global||^2
            prox_term = 0.0
            for p_local, p_global in zip(model.parameters(), global_model.parameters()):
                prox_term += ((p_local - p_global.detach()) ** 2).sum()
            loss += (mu / 2) * prox_term

            loss.backward()
            optimizer.step()
    return model

# ── EVALUATE ─────────────────────────────────────────────────────────────────
def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        Xt   = torch.tensor(X, dtype=torch.float32).to(DEVICE)
        prob = model(Xt).cpu().numpy()
    pred = (prob >= 0.5).astype(int)
    auc  = roc_auc_score(y, prob)
    f1   = f1_score(y, pred, zero_division=0)
    rec  = recall_score(y, pred, zero_division=0)
    return auc, f1, rec

# ── FEDERATED TRAINING ────────────────────────────────────────────────────────
print(f"\nStarting FedProx FL (μ={MU}): {FL_ROUNDS} rounds\n")
global_model = FraudNet(N_FEAT).to(DEVICE)

for rnd in range(1, FL_ROUNDS + 1):
    local_weights = []
    local_sizes   = []

    for bank_id, (X_b, y_b) in enumerate(bank_data):
        # SMOTE per bank
        try:
            sm      = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(3, int(y_b.sum())-1))
            X_r, y_r = sm.fit_resample(X_b, y_b)
        except Exception:
            X_r, y_r = X_b, y_b

        local_model = copy.deepcopy(global_model)
        loader      = make_loader(X_r, y_r)
        local_model = local_train_fedprox(local_model, global_model, loader)

        local_weights.append(copy.deepcopy(local_model.state_dict()))
        local_sizes.append(len(X_b))

    # FedAvg aggregation of weights
    total = sum(local_sizes)
    new_state = copy.deepcopy(local_weights[0])
    for key in new_state:
        new_state[key] = sum(
            local_weights[i][key] * (local_sizes[i] / total)
            for i in range(N_BANKS)
        )
    global_model.load_state_dict(new_state)

    if rnd % 5 == 0 or rnd == 1:
        auc, f1, rec = evaluate(global_model, X_test, y_test)
        print(f"  Round {rnd:>3} | AUC: {auc:.4f} | F1: {f1:.4f} | Recall: {rec:.4f}")

# ── FINAL EVALUATION ──────────────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL FEDPROX MODEL EVALUATION")
print("="*60)
auc, f1, rec = evaluate(global_model, X_test, y_test)
print(f"AUC-ROC  : {auc:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"Recall   : {rec:.4f}")

# Recall at 1% FPR (your paper's metric)
global_model.eval()
with torch.no_grad():
    prob = global_model(torch.tensor(X_test, dtype=torch.float32).to(DEVICE)).cpu().numpy()

from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_test, prob)
idx_1pct = np.where(fpr <= 0.01)[0][-1]
print(f"Recall @ 1% FPR: {tpr[idx_1pct]:.4f}  (threshold={thresholds[idx_1pct]:.4f})")

# Per-bank fairness
print("\nPer-Bank AUC (Fairness):")
bank_aucs = []
for i, (X_b, y_b) in enumerate(bank_data):
    if len(np.unique(y_b)) < 2: continue
    auc_b, _, _ = evaluate(global_model, X_b, y_b)
    bank_aucs.append(auc_b)
    print(f"  Bank {i+1}: {auc_b:.4f}")
print(f"  σ_AUC = {np.std(bank_aucs):.4f}")

Device: cuda
Loading dataset...

Non-IID split across 5 banks:
  Bank 1: 45,599 records | fraud 0.24%
  Bank 2: 45,672 records | fraud 0.40%
  Bank 3: 45,490 records | fraud 0.00%
  Bank 4: 45,592 records | fraud 0.22%
  Bank 5: 45,491 records | fraud 0.00%

Starting FedProx FL (μ=0.01): 20 rounds

  Round   1 | AUC: 0.9737 | F1: 0.6935 | Recall: 0.7041
  Round   5 | AUC: 0.9760 | F1: 0.5372 | Recall: 0.8469
  Round  10 | AUC: 0.9768 | F1: 0.6946 | Recall: 0.8469
  Round  15 | AUC: 0.9763 | F1: 0.6776 | Recall: 0.8469
  Round  20 | AUC: 0.9746 | F1: 0.7330 | Recall: 0.8265

FINAL FEDPROX MODEL EVALUATION
AUC-ROC  : 0.9746
F1 Score : 0.7330
Recall   : 0.8265
Recall @ 1% FPR: 0.8980  (threshold=0.2869)

Per-Bank AUC (Fairness):
  Bank 1: 0.9974
  Bank 2: 0.9971
  Bank 4: 0.9949
  Bank 5: 1.0000
  σ_AUC = 0.0018
